# SOMAX: Training & Evaluation

**Step 2 of 2 — Benchmarking, training, and GGUF export**

**Prerequisite:** Run `scripts/download.py` first to download the WAXAL dataset.

**Pipeline:**
1. Setup & Dependencies
2. Baseline Token Fertility
3. Train Router Classifier
4. Train Unified BPE Vocabulary
5. Train LoRA Variant (A–E)
6. Benchmark Fertility Reduction
7. Export to GGUF

## 1. Setup & Dependencies

In [23]:
from huggingface_hub import notebook_login

notebook_login()

In [24]:
# Check GPU
!nvidia-smi

Fri Apr 10 20:04:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   69C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Skip torch and psutil (already in Colab)
# Focus on the research-specific extensions
!uv pip install -U transformers datasets accelerate peft bitsandbytes llama-cpp-python

Using Python 3.12.13 environment at: /usr
Resolved 78 packages in 1.56s                                        
Prepared 41 packages in 8m 37s                                           
Uninstalled 23 packages in 1.12s
Installed 41 packages in 442ms.20                           
 - aiohttp==3.13.4
 + aiohttp==3.13.5
 + bitsandbytes==0.49.2
 - charset-normalizer==3.4.6
 + charset-normalizer==3.4.7
 - click==8.3.1
 + click==8.3.2
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-pathfinder==1.5.0
 + cuda-pathfinder==1.5.2
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - datasets==4.0.0
 + datasets==4.8.4
 - dill==0.3.8
 + dill==0.4.1
 + diskcache==5.6.3
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 - hf-xet==1.4.2
 + hf-xet==1.4.3
 - huggingface-hub==1.8.0
 + huggingface-hub==1.10.1
 + llama-cpp-python==0.3.20
 - multiprocess==0.70.16
 + multiprocess==0.70.19
 - numpy==2.0.2
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvi

Using Python 3.12.13 environment at: /usr
Resolved 78 packages in 297ms                                        
Checked 78 packages in 1ms


In [ ]:
import os
from pathlib import Path

if not os.path.exists('somax'):
    !git clone 'https://github.com/attabeezy/somax.git'
    %cd somax
else:
    %cd somax

Cloning into 'somax'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 111 (delta 54), reused 89 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 69.18 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/content/somax


In [ ]:
import os

# ── Configuration ─────────────────────────────────────────
LANGUAGE    = "akan"                      # proof-of-concept language
TRAIN_GROUP = "D"                         # primary hypothesis: TTS → ASR → TTS
BASE_MODEL  = "meta-llama/Llama-3.2-1B"
DATA_DIR    = f"data/{LANGUAGE}"
CONFIG_FILE = "configs/variants.yaml"
# ──────────────────────────────────────────────────────────

HF_TOKEN = os.environ.get("HF_TOKEN", "SIKE TOKEN NOT SET")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Required for Llama models.")
    print("Set it with:  os.environ['HF_TOKEN'] = 'your_token'")

In [ ]:
import shutil

def setup_drive() -> str | None:
    """Mount Google Drive and return results base path."""
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=120000)
        base = "/content/drive/MyDrive/WAXAL-results"
        os.makedirs(base, exist_ok=True)
        print(f"Google Drive mounted: {base}")
        return base
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str | None, label: str) -> None:
    """Copy a local directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest = os.path.join(drive_base, label)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(source_dir, dest)
    print(f"Saved to Drive: {dest}")

DRIVE_BASE = setup_drive()

# Download dataset if not present
if not Path(DATA_DIR).exists() or not list(Path(DATA_DIR).glob("*.jsonl")):
    print("Dataset not found. Downloading WAXAL dataset...")
    !python scripts/download.py --lang {LANGUAGE} --output data/
else:
    print(f"Dataset found: {DATA_DIR}")

Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive/WAXAL-results
Dataset not found. Downloading WAXAL dataset...
Target directory: data/akan

ASR config: aka_asr
README.md: 29.2kB [00:00, 15.2MB/s]
Resolving data files: 100% 72/72 [00:00<00:00, 19560.20it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 19942.28it/s]
  train: 10107 samples -> data/akan/aka_asr_train.jsonl
Resolving data files: 100% 72/72 [00:00<00:00, 31033.80it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 32653.67it/s]
  validation: 1123 samples -> data/akan/aka_asr_validation.jsonl
Resolving data files: 100% 72/72 [00:00<00:00, 31284.56it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 33351.85it/s]
  test: 1522 samples -> data/akan/aka_asr_test.jsonl

TTS config: twi_tts
Resolving data files: 100% 72/72 [00:00<00:00, 31017.86it/s]
  train: 872 samples -> data/akan/twi_tts_train.jsonl
Resolving data files: 100% 72/72 [00:00<00:00, 31778.37it/s]
  validation: 117 samples -> data/akan/

## 2. Baseline Token Fertility

In [ ]:
!hf auth login --token YOUR_HF_TOKEN

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `somax` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `somax`


In [ ]:
!pip install torchvision --upgrade --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 75.0 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128


In [ ]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --output results/baseline_fertility.json

Loaded 104 samples from data/akan/twi_tts_test.jsonl
Loading HuggingFace tokenizer: meta-llama/Llama-3.2-1B
tokenizer_config.json: 50.5kB [00:00, 73.3MB/s]
tokenizer.json: 9.09MB [00:00, 25.4MB/s]
special_tokens_map.json: 100% 301/301 [00:00<00:00, 2.02MB/s]

[Baseline] Tokenizer: meta-llama/Llama-3.2-1B
  Token Fertility (F): 2.810 ± 0.416 tokens/word
  Total tokens: 22374  |  Total words: 7961
  Samples: 104

Results saved to: results/baseline_fertility.json


## 3. Train Router Classifier

In [ ]:
!python scripts/train_router.py \
    --data {DATA_DIR} \
    --output models/router/ \
    --language {LANGUAGE}

Training router classifier for language: akan
Data: data/akan
Loaded 10107 ASR samples
Loaded 872 TTS samples

Total samples: 10979 (10107 ASR, 872 TTS)
Training classifier...
Cross-val F1 (5-fold): 0.914 ± 0.027

Classifier saved to: models/router/akan_router.pkl

Sanity check:
  'uhm chale me dwo o' -> logic
  'The president delivered a formal address to the assembly' -> robust


## 4. Train Unified BPE Vocabulary

In [ ]:
!python scripts/train_bpe.py \
    --input {DATA_DIR} \
    --output models/tokenizers/ \
    --language {LANGUAGE} \
    --vocab-size 8000

ASR: 10107 samples from data/akan/aka_asr_train.jsonl
TTS: 872 samples from data/akan/twi_tts_train.jsonl
Training unified BPE on 10979 samples (10107 ASR + 872 TTS)...
[00:00:00] Tokenize words                 ██████████████████ 22215    /    22215[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 22215    /    22215
[00:00:00] Compute merges                 ██████████████████ 7889     /     7889
Unified tokenizer saved to: models/tokenizers/akan/unified_tokenizer.json
Tokenizer config saved to: models/tokenizers/akan/tokenizer_config.json
Computing per-stream token statistics...
Stream token stats saved to: models/tokenizers/akan/stream_token_stats.json

Training stats saved to: models/tokenizers/akan/training_stats.json
Vocab size: 8000
Token dominance: {'asr': 5308, 'tts': 1403, 'shared': 1289}
Done.


## 5. Train LoRA Variant

Edit `TRAIN_GROUP` in the config cell above to run a different variant:

| Group | Sequence | Notes |
|-------|----------|-------|
| A | ASR only | |
| B | TTS only | |
| C | Mixed | |
| **D** | TTS → ASR → TTS | Primary hypothesis |
| E | ASR → TTS | |

In [ ]:
!python scripts/train_lora.py \
    --group {TRAIN_GROUP} \
    --model {BASE_MODEL} \
    --data {DATA_DIR} \
    --output checkpoints/ \
    --config {CONFIG_FILE}

Variant:    D
Base model: meta-llama/Llama-3.2-1B
Data:       data/akan
Output:     checkpoints/variant_D
Stages:     3

Loading model: meta-llama/Llama-3.2-1B
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 2.47G/2.47G [00:27<00:00, 90.1MB/s]
Loading weights: 100% 146/146 [00:02<00:00, 63.93it/s]
generation_config.json: 100% 185/185 [00:00<00:00, 1.06MB/s]
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(nam

In [ ]:
save_to_drive("checkpoints", DRIVE_BASE, "checkpoints")

## 6. Benchmark Fertility Reduction

In [ ]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --waxal-tokenizer models/tokenizers/{LANGUAGE}/unified_tokenizer.json \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --compare \
    --output results/fertility_comparison.json

## 7. Export to GGUF (for Edge Deployment)

In [ ]:
!python scripts/export_gguf.py \
    --checkpoint checkpoints/variant_{TRAIN_GROUP}/final/ \
    --output models/gguf/ \
    --quantization Q4_K_M

In [ ]:
save_to_drive("results", DRIVE_BASE, "results")
save_to_drive("models", DRIVE_BASE, "models")

## Summary

In [ ]:
import json
from pathlib import Path

print("=" * 50)
print("WAXAL-Dual-Core Pipeline Complete!")
print("=" * 50)
print(f"Language:       {LANGUAGE}")
print(f"Variant:        {TRAIN_GROUP}")

results_path = Path("results/fertility_comparison.json")
if results_path.exists():
    r = json.loads(results_path.read_text())
    print(f"\nBaseline F:     {r['baseline']['fertility_mean']:.3f} tokens/word")
    print(f"WAXAL F:        {r['waxal']['fertility_mean']:.3f} tokens/word")
    print(f"Reduction:      {r['reduction_pct']:.1f}%  (target: ≥30%)")
else:
    baseline_path = Path("results/baseline_fertility.json")
    if baseline_path.exists():
        r = json.loads(baseline_path.read_text())
        print(f"\nBaseline F:     {r['fertility_mean']:.3f} tokens/word")

print("\nNext steps:")
print("  1. Download results from Google Drive")
print("  2. Run scripts/benchmark_inference.py on Dell Latitude 7400")
print("  3. Run scripts/benchmark_fertility.py --compare on edge hardware")